# Project Overview - Batch Ingestion

This project focuses on building a batch data pipeline in Databricks using a retail sales scenario. 

It involves creating an organized directory structure in the Databricks File System (DBFS) and generating batch sales data from multiple stores in different formats—CSV for Store A and JSON for Store B. 

The data is ingested into Spark DataFrames and stored as Delta tables to enable structured querying and analytics. Finally, data from all sources is consolidated into a single table to provide a unified view of sales.

The project reflects real-world data engineering practices, including handling heterogeneous data sources, standardizing data formats, and preparing consolidated datasets for downstream analytics and reporting using Databricks, Spark, and SQL.

## Step 1: Create Directories in DBFS

In [0]:
path = "dbfs:/FileStore/tables"
directory_path = "dbfs:/FileStore/tables/stores"
store_path = "dbfs:/storesData"
catalog_name = "catalog_revolution"
schema_name = "schema_lakehouse_revolution"



In [0]:
# Delete if it exists
try:
    dbutils.fs.rm(path, recurse=True)
    print(f"Deleted: {path}")
except Exception:
    print(f"Path does not exist, skipping delete: {path}")

# Recreate folders
dbutils.fs.mkdirs(path)
dbutils.fs.mkdirs(f"{path}/stores")

# Verify
dbutils.fs.ls("dbfs:/FileStore")
dbutils.fs.ls(path)


In [0]:
if not any(file.name == "stores/" for file in dbutils.fs.ls("dbfs:/FileStore/tables/")):
  dbutils.fs.mkdirs(directory_path)
dbutils.fs.ls("dbfs:/FileStore/tables")


if not any(file.name == "storesData/" for file in dbutils.fs.ls("dbfs:/")):
  dbutils.fs.mkdirs(store_path)
dbutils.fs.ls('dbfs:/')

## Step 2: Write Sample Data for Store A to CSV

In [0]:
import pandas as pd
import json

In [0]:
store_a_data = {
"ProductID": [101, 102, 103, 104],
"ProductName": ["Widget", "Gadget", "Thingamajig", "Doohickey"],
"QuantitySold": [15, 25, 35, 45],
"SalesAmount": [150.0, 250.0, 350.0, 450.0],
"TransactionDate": ["2023-01-01", "2023-01-02", "2023-01-03", "2023-01-04"]
}

store_a_df = pd.DataFrame(store_a_data)
csv_data = store_a_df.to_csv(index=False)
dbutils.fs.put(f"{store_path}/store_a.csv", csv_data, overwrite=True)

dbutils.fs.ls(store_path)

## Step 3: Write Sample Data for Store B to JSON

In [0]:
store_b_data = [
{"ProductID": 201, "ProductName": "Gizmo", "QuantitySold": 10, "SalesAmount": 100.0, "TransactionDate": "2023-01-01"},
{"ProductID": 202, "ProductName": "Contraption", "QuantitySold": 20, "SalesAmount": 200.0, "TransactionDate": "2023-01-02"}
]

json_data = json.dumps(store_b_data, indent=4)
dbutils.fs.put(f"{store_path}/store_b.json", json_data, overwrite=True)

dbutils.fs.ls(store_path)


## Step 4: Load Data into Spark DataFrames and Save As Delta Tables

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import *

In [0]:
# spark.sql("CREATE CATALOG IF NOT EXISTS lakehouse_revolution")
# spark.sql(f"USE CATALOG {catalog_name}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema_name}_ex1")
spark.sql(f"USE {schema_name}_ex1")

In [0]:
store_a_df = (
    spark.read.format('csv').option('header','true')
    .load(f"{store_path}/store_a.csv")
)
display(store_a_data)

store_b_df =(
    spark
        .read
        .format('json')
        .option("multiline", "true")
        .schema("ProductID INT, ProductName STRING, QuantitySold INT, SalesAmount DOUBLE, TransactionDate STRING")
        .load(f"{store_path}/store_b.json")
)
display(store_b_df)

In [0]:
# save data frames
store_a_df.write.format('delta').mode('overwrite').saveAsTable(f"{schema_name}_ex1.store_a")
store_b_df.write.format('delta').mode('overwrite').saveAsTable(f"{schema_name}_ex1.store_b")


## Step 5: Query the Delta Tables


In [0]:
spark.sql(f"select * from {schema_name}_ex1.store_a").show()
spark.sql(f"select * from {schema_name}_ex1.store_b").show()


Step 6: Merge Data into a New Table

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {schema_name}_ex1.consolidated_sales")

create_table = (
    f"""
    CREATE TABLE IF NOT EXISTS {schema_name}_ex1.consolidated_sales AS
    SELECT * FROM {schema_name}_ex1.store_a
    UNION ALL
    SELECT * FROM {schema_name}_ex1.store_b
    """
)
spark.sql(create_table)


Step 7: Verify Consolidated Data

In [0]:
spark.sql(f"SELECT * FROM {schema_name}_ex1.consolidated_sales").show()